<a href="https://colab.research.google.com/github/gmauricio-toledo/NLP-LCC/blob/main/Notebooks/07-WordEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Word and Document Embeddings</h1>

En esta notebook exploraremos el uso de distintos embeddings para resolver.algunas tareas del NLP.

Los puntos principales de esta notebook son:

In [ ]:
import nltk
import pandas as pd
import numpy as np
from nltk import word_tokenize
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import matplotlib.pyplot as plt
from string import punctuation

nltk.download('punkt_tab')
nltk.download('stopwords')

stopwords = nltk.corpus.stopwords.words('spanish')

In [ ]:
!pip install -qq umap-learn
!pip install gensim

In [ ]:
def normalizar_vector(v):
    if np.linalg.norm(v) == 0:
        return v
    else:
        return v / np.linalg.norm(v)

# Exploración word2vec

Usaremos la implementación de gensim: https://radimrehurek.com/gensim/models/word2vec.html.

Tensorflow también tiene una implementación ([tutorial](https://www.tensorflow.org/text/tutorials/word2vec)).

El artículo original: https://arxiv.org/pdf/1301.3781

## Preprocesamiento y limpieza del texto

In [ ]:
url = "https://raw.githubusercontent.com/gmauricio-toledo/NLP-MCD/main/data/spanish-wikipedia-dataframe.csv"
df = pd.read_csv(url,index_col=0)
df.drop(columns='doc_id',inplace=True)
df

In [ ]:
docs_raw = df['Texto'].tolist()
docs = [re.sub(r'\d+', ' ', doc) for doc in docs_raw]
tokenized_docs = [word_tokenize(doc) for doc in docs]
docs = [' '.join(doc) for doc in tokenized_docs]
docs[:3]

## Usar un modelo pre-entrenado

### Un modelo de gensim

Gensim tiene varios modelos de word2vec preentrenados:

In [ ]:
import gensim.downloader

for x in gensim.downloader.info()['models'].keys():
    print(x)

Descarguemos alguno de estos modelos.

Tarda alrededor de 20 minutos

In [ ]:
import gensim.downloader

# pt_w2v_model = gensim.downloader.load('glove-wiki-gigaword-50')
pt_w2v_model = gensim.downloader.load('word2vec-google-news-300')

Obtengamos los vectores

In [ ]:
vectors = pt_w2v_model.vectors
vectors.shape

Realicemos algunas tareas de similitud:

Resolvamos algunas analogías:

### Un modelo *externo*

Descarguemos un modelo externo y experimentemos con él

In [ ]:
!gdown 0B7XkCwpI5KDYNlNUTTlSS21pQmM

Leamos el modelo, **tarda alrededor de 2 minutos**

In [ ]:
from gensim.models import KeyedVectors

pretrained_model_path = 'GoogleNews-vectors-negative300.bin.gz'

pt_w2v_model = KeyedVectors.load_word2vec_format(pretrained_model_path, binary=True)

### Experimentación

Dimensión de los embeddings

In [ ]:
pt_w2v_model.vector_size

In [ ]:
vocabulary = pt_w2v_model.index_to_key
print(f"Tamaño del vocabulario: {len(vocabulary)}")
print(vocabulary[:20])

In [ ]:
word = "king"

pt_w2v_model.most_similar(word,topn=15)

In [ ]:
pt_w2v_model.most_similar(positive=['woman', 'king'], negative=['man'], topn=5)
# pt_w2v_model.most_similar(positive=['woman', 'actor'], negative=['man'], topn=5)

Veamos la similitud coseno entre palabras _similares_ y _no similares_

In [ ]:
# Palabras no similares
word1 = "dream"
word2 = "technology"
similarity1 = pt_w2v_model.similarity(word1, word2)
print(similarity1)

# Palabras relativamente similares
word5 = "computer"
word6 = "pencil"
similarity3 = pt_w2v_model.similarity(word5, word6)
print(similarity3)

# Palabras muy similares
word3 = "boy"
word4 = "girl"
similarity2 = pt_w2v_model.similarity(word3, word4)
print(similarity2)

Exploremos la geometría de estos embeddings:

In [ ]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
word = 'spain'

word_vector = pt_w2v_model[word]

dim = pt_w2v_model.vector_size
print(f"palabra: {word}")
zeros = np.where(word_vector==0)[0].shape[0]
print(f"Número de entradas 0: {zeros}/{dim}={round(100*zeros/dim,2)}%")

In [ ]:
pt_w2v_model['spain']

Veamos las direcciones entre pares de palabras, las cuales codifican la relación semántica entre ellas

In [ ]:
word1 = 'sweden'
word2 = 'stockholm'
word3 = 'france'
word4 = 'paris'
word5 = 'germany'
word6 = 'berlin'

w1 = pt_w2v_model[word1]
w2 = pt_w2v_model[word2]
w3 = pt_w2v_model[word3]
w4 = pt_w2v_model[word4]
w5 = pt_w2v_model[word5]
w6 = pt_w2v_model[word6]

print(f"Palabras a analizar:\n{word1}-{word2}\n{word3}-{word4}\n{word5}-{word6}")
dif_1, dif_2, dif_3 = w1 - w2, w3 - w4, w5 - w6
print("Similitud entre las diferencias:")
print(cosine_similarity(dif_1, dif_2),cosine_similarity(dif_1, dif_3), cosine_similarity(dif_2, dif_3))

# Ahora con palabras aleatorias:
six_random_words = np.random.choice(vocabulary,size=6,replace=False)
print(f"\nEl mismo análisis con 6 palabras aleatorias:\n{six_random_words}")
rw1 = pt_w2v_model[six_random_words[0]]
rw2 = pt_w2v_model[six_random_words[1]]
rw3 = pt_w2v_model[six_random_words[2]]
rw4 = pt_w2v_model[six_random_words[3]]
rw5 = pt_w2v_model[six_random_words[4]]
rw6 = pt_w2v_model[six_random_words[5]]

dif_1, dif_2, dif_3 = rw1 - rw2, rw3 - rw4, rw5 - rw6
print("Similitud entre las diferencias:")
print(cosine_similarity(dif_1, dif_2),cosine_similarity(dif_1, dif_3), cosine_similarity(dif_2, dif_3))

## Entrenar un modelo en el corpus

Es importante considerar que el modelo depende mucho de los datos con los que se entrena. Para muchas tareas generales basta con utilizar un modelo preentrenado, pero algunas aplicaciones específicas (por ejemplo, específicas de un dominio especializado) pueden requerir entrenar un modelo en un corpus específico.

In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=5, workers=4)

Veamos el vocabulario obtenido

In [ ]:
vocabulary = w2v_model.wv.index_to_key
print(f"Tamaño del vocabulario: {len(vocabulary)}")
print(vocabulary[:20])

Los vectores:

In [ ]:
word_vectors = w2v_model.wv.vectors
word_vectors.shape

In [ ]:
# Guardar todo el modelo
w2v_model.save("word2vec.model")

# Guardar sólo los vectores
w2v_model.wv.save("word2vec.wordvectors")

In [ ]:
word = 'amplia'

word_vector = w2v_model.wv[word]

dim = w2v_model.wv.vector_size
print(f"palabra: {word}")
zeros = np.where(word_vector==0)[0].shape[0]
print(f"Número de entradas 0: {zeros}/{dim}={round(100*zeros/dim,2)}%")

Veamos qué pasa con las palabras **OOV**

In [ ]:
w2v_model.wv['holonomia']

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,1))
plt.suptitle("Word2Vec")
plt.imshow(word_vector.reshape(1,-1))
plt.yticks([])
plt.show()

plt.figure(figsize=(10,3))
plt.suptitle("BOW")
plt.imshow(bow_vector.reshape(27,-1))
plt.yticks([])
plt.show()

In [ ]:
#@title Grafiquemos la reducción de dimensionalidad 3d t-SNE

# from sklearn.manifold import TSNE
# import plotly.graph_objects as go
# import plotly

# vocabulary = w2v_model.wv.index_to_key
# word_vectors = w2v_model.wv.vectors

# tsne = TSNE(n_components=3, metric='cosine')
# X_tsne = tsne.fit_transform(word_vectors)

# plotly.offline.init_notebook_mode()

# trace = go.Scatter3d(
#     x=X_tsne[:,0],
#     y=X_tsne[:,1],
#     z=X_tsne[:,2],
#     mode='markers',
#     marker={
#         'size': 3,
#         'opacity': 0.75,
#         'color': 'black'
#     },
#     hovertemplate='%{text}<extra></extra>',
#     text = [f"{vocabulary[j]}" for j in range(X_tsne.shape[0])]
# )

# layout = go.Layout(
#     margin={'l': 0, 'r': 0, 'b': 0, 't': 0}
# )

# data = [trace]

# plot_figure = go.Figure(data=data, layout=layout)

# plot_figure.update_layout(
#     title = 'Wikipedia Words',
#     scene = dict(
#         xaxis = dict(visible=False),
#         yaxis = dict(visible=False),
#         zaxis =dict(visible=False)
#         )
#     )

# plotly.offline.plot(plot_figure, filename='/content/drive/MyDrive/Colab Notebooks/NLP/Mi curso/wiki-w2v-tsne3d-words.html')

## Vectores de documentos

Algunas técnicas para obtener vectores de documentos:

* Promediar los vectores de word2vec. Según Le y Mikolov, este enfoque no funciona bien para tareas de análisis de sentimientos, porque «pierde el orden de las palabras del mismo modo que los modelos estándar de bolsa de palabras» y «no reconoce muchos fenómenos lingüísticos sofisticados, como el sarcasmo». Por otro lado, según Kenter et al. 2016, «promediar simplemente las incrustaciones de palabras de todas las palabras de un texto ha demostrado ser una línea de base o característica sólida en multitud de tareas», como las tareas de similitud de textos cortos.
* Ponderar los vectores de palabras con su TF-IDF para disminuir la influencia de las palabras más comunes.
* Concatenar los vectores de palabras.

Observar que la operación de sumar vectores ignora el orden de las palabras por lo que caemos en una representación tipo BOW.

Gensim permite obtener un promedio de vectores

In [ ]:
doc_vectors = np.array([w2v_model.wv.get_mean_vector(doc) for doc in docs])
doc_vectors.shape

In [ ]:
plt.figure()
plt.suptitle("Normas de los vectores de documentos")
plt.hist(np.linalg.norm(doc_vectors,axis=1))
plt.show()

In [ ]:
np.save('wikipedia_w2v_mean_doc_vectors.npy',doc_vectors)

In [ ]:
doc_vectors = np.load('wikipedia_w2v_mean_doc_vectors.npy')

In [ ]:
for i,z in enumerate(doc_vectors):
    doc_vectors[i] = normalizar_vector(z)

In [ ]:
#@title Grafiquemos la reducción de dimensionalidad 3d UMAP

# from umap import UMAP
# import matplotlib.pyplot as plt
# import plotly
# import plotly.graph_objs as go

# umap = UMAP(metric='cosine',n_components=3)
# X_umap = umap.fit_transform(doc_vectors)

# plotly.offline.init_notebook_mode()

# trace = go.Scatter3d(
#     x=X_umap[:,0],
#     y=X_umap[:,1],
#     z=X_umap[:,2],
#     mode='markers',
#     marker={
#         'size': 3,
#         'opacity': 0.75,
#         'color': 'blue'
#     },
#     hovertemplate='%{text}<extra></extra>',
#     text = [f"{docs_raw[j][:75]}" for j in range(X_umap.shape[0])]
# )

# layout = go.Layout(
#     margin={'l': 0, 'r': 0, 'b': 0, 't': 0}
# )

# data = [trace]

# plot_figure = go.Figure(data=data, layout=layout)

# plot_figure.update_layout(
#     title = 'Wikipedia Docs',
#     scene = dict(
#         xaxis = dict(visible=False),
#         yaxis = dict(visible=False),
#         zaxis =dict(visible=False)
#         )
#     )

# plotly.offline.plot(plot_figure, filename='/content/drive/MyDrive/Colab Notebooks/NLP/Mi curso/wiki-w2v-norm-umap3d-docs.html')

# Uso como features

| Modelo   | Granularidad       | Ventaja clave                          |
|----------|--------------------|----------------------------------------|
| Word2Vec | palabra            | rápido, baseline sólido                |
| FastText | subpalabra (n-gramas) | maneja OOV y morfología             |
| GloVe    | palabra            | se entrena sobre co-ocurrencias de todo el corpus a la vez en lugar de ventanas |
| Doc2Vec  | párrafo/doc        | vector para texto completo             |

## Clasificación

In [ ]:
stopwords = nltk.corpus.stopwords.words('english')

In [ ]:
!gdown 1uKAlVHIUelY1C0DLgc8arLLhoAIwWmdX

In [ ]:
import pandas as pd

yt_df = pd.read_csv('YoutubeCommentsDataSet.csv')
yt_df.dropna(inplace=True)
display(yt_df)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt


y = LabelEncoder().fit_transform(yt_df['Sentiment'].values)

plt.figure()
plt.hist(y)
plt.xticks([0,1,2],labels=['Negative','Neutral','Positive'])
plt.show()

Dividimos en train/test

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(yt_df['Comment'].values, y,
                                                            test_size=0.25,
                                                            stratify=y, # IMPORTANTE
                                                            random_state=642)

In [ ]:
plt.figure()
plt.subplot(1,2,1)
plt.hist(y_train)
plt.title("Train labels")
plt.xticks([0,1,2],labels=['Negative','Neutral','Positive'])
plt.subplot(1,2,2)
plt.hist(y_test)
plt.title("Test labels")
plt.xticks([0,1,2],labels=['Negative','Neutral','Positive'])
plt.show()

In [ ]:
import numpy as np

random_idxs = np.random.choice(yt_df.shape[0],5,replace=False)

for j in random_idxs:
    text = yt_df.loc[j,'Comment']
    sentiment = yt_df.loc[j,'Sentiment']
    print(f"{text[:80]}...:\n\t{sentiment}")


Preprocesamiento y limpieza

In [ ]:
import re

X_train_raw = [re.sub(r'\d+', ' ', doc) for doc in X_train_raw]
train_tokenized_docs = [[x for x in word_tokenize(doc) if x not in stopwords and x not in punctuation]
                        for doc in X_train_raw]
print(f"Número de documentos: {len(train_tokenized_docs)}")
train_tokenized_docs = [D for D in train_tokenized_docs if len(D)!=0]
print(f"Número de documentos: {len(train_tokenized_docs)}")
train_docs = [' '.join(doc) for doc in train_tokenized_docs]

X_test_raw = [re.sub(r'\d+', ' ', doc) for doc in X_test_raw]
test_tokenized_docs = [[x for x in word_tokenize(doc) if x not in stopwords and x not in punctuation]
                       for doc in X_test_raw]
print(f"Número de documentos: {len(test_tokenized_docs)}")
test_tokenized_docs = [D for D in test_tokenized_docs if len(D)!=0]
print(f"Número de documentos: {len(test_tokenized_docs)}")
test_docs = [' '.join(doc) for doc in test_tokenized_docs]

In [ ]:
vectorizer = TfidfVectorizer(max_features=2000, stop_words=stopwords)
X_train_tfidf = vectorizer.fit_transform(train_docs).toarray()
X_test_tfidf = vectorizer.transform(test_docs).toarray()

### Entrenando nuestro modelo de embeddings

Entrenemos un modelo de word2vec en el corpus IMDB. **Tarda alrededor de 2 minutos**

In [ ]:
from gensim.models import Word2Vec

w2v_20ng_model = Word2Vec(sentences=train_tokenized_docs, vector_size=100, window=3, min_count=3, workers=4)

In [ ]:
word = 'guys'
w2v_20ng_model.wv[word]

In [ ]:
w2v_20ng_model.wv.most_similar(word,topn=10)

In [ ]:
text = test_tokenized_docs[0]
print(text)
w2v_20ng_model.wv.get_mean_vector(text)

In [ ]:
train_doc_vectors = np.zeros((len(train_docs), w2v_20ng_model.wv.vector_size))
test_doc_vectors = np.zeros((len(test_docs), w2v_20ng_model.wv.vector_size))

train_zero_vector_num = 0
test_zero_vector_num = 0

for i, doc in enumerate(train_tokenized_docs):
    try:
        train_doc_vectors[i] = w2v_20ng_model.wv.get_mean_vector(doc)
    except:
        train_zero_vector_num += 1

for i, doc in enumerate(test_tokenized_docs):
    try:
        test_doc_vectors[i] = w2v_20ng_model.wv.get_mean_vector(doc)
    except:
        test_zero_vector_num += 1

In [ ]:
print(f"Vectores nulos en el conjunto train: {train_zero_vector_num}")
print(f"Vectores nulos en el conjunto test: {test_zero_vector_num}")

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

models = [SVC(),
          LogisticRegression(),
          MLPClassifier(hidden_layer_sizes=(50)),
          KNeighborsClassifier(metric='cosine')
        ]

model_names = ['SVC','LR','MLP','KNN']
f1_scores_1 = []

for name, model in zip(model_names,models):
    model.fit(train_doc_vectors, y_train)
    train_predictions = model.predict(train_doc_vectors)
    test_predictions = model.predict(test_doc_vectors)
    train_f1 = f1_score(y_train,train_predictions,average='macro')
    test_f1 = f1_score(y_test,test_predictions,average='macro')
    print(f"{name}:\n\tTrain: {round(train_f1,3)}\n\tTest: {round(test_f1,3)}")
    f1_scores_1.append((train_f1,test_f1))

### Usando un modelo pre-entrenado

In [ ]:
!gdown 1k9bBE7ViPU8YxrbHiunns926zx-lip0i

Tarda alrededor de 1 minuto en leer el modelo

In [ ]:
from gensim.models import KeyedVectors

pretrained_model_path = 'GoogleNews-vectors-negative300.bin.gz'

pt_w2v_model = KeyedVectors.load_word2vec_format(pretrained_model_path, binary=True)

In [ ]:
pt_w2v_model.vector_size

In [ ]:
vector_size = pt_w2v_model.vector_size
train_doc_vectors = np.zeros((len(train_docs), vector_size))
test_doc_vectors = np.zeros((len(test_docs), vector_size))

zero_train_vector_idxs = []

train_zero_vector_num = 0
test_zero_vector_num = 0

for i, doc in enumerate(train_tokenized_docs):
    try:
        train_doc_vectors[i] = pt_w2v_model.get_mean_vector(doc)
    except:
        train_zero_vector_num += 1
        zero_train_vector_idxs.append(i)

for i, doc in enumerate(test_tokenized_docs):
    try:
        test_doc_vectors[i] = pt_w2v_model.get_mean_vector(doc)
    except:
        test_zero_vector_num += 1

In [ ]:
print(f"Vectores nulos en el conjunto train: {train_zero_vector_num}")
print(f"Vectores nulos en el conjunto test: {test_zero_vector_num}")

In [ ]:
train_doc_vectors[:4,:5]

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

models = [SVC(),
          LogisticRegression(),
          MLPClassifier(hidden_layer_sizes=(50)),
          KNeighborsClassifier(metric='cosine')
        ]

model_names = ['SVC','LR','MLP','KNN']
f1_scores_2 = []

print("Pre-trained Word2vec vectors")
for name, model in zip(model_names,models):
    model.fit(train_doc_vectors, y_train)
    train_predictions = model.predict(train_doc_vectors)
    test_predictions = model.predict(test_doc_vectors)
    train_f1 = f1_score(y_train,train_predictions,average='macro')
    test_f1 = f1_score(y_test,test_predictions,average='macro')
    print(f"{name}:\n\tTrain: {round(train_f1,3)}\n\tTest: {round(test_f1,3)}")
    f1_scores_2.append((train_f1,test_f1))

### Baseline

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

models = [SVC(),
          LogisticRegression(),
          MLPClassifier(hidden_layer_sizes=(50)),
          KNeighborsClassifier(metric='cosine')
        ]

model_names = ['SVC','LR','MLP','KNN']
f1_scores_3 = []

print("TF-IDF Baseline")
for name, model in zip(model_names,models):
    model.fit(X_train_tfidf, y_train)
    train_predictions = model.predict(X_train_tfidf)
    test_predictions = model.predict(X_test_tfidf)
    train_f1 = f1_score(y_train,train_predictions,average='macro')
    test_f1 = f1_score(y_test,test_predictions,average='macro')
    print(f"{name}:\n\tTrain: {round(train_f1,3)}\n\tTest: {round(test_f1,3)}")
    f1_scores_3.append((train_f1,test_f1))

### Comparación

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_names = ['SVC', 'Log Reg', 'MLP', 'KNN']
approaches = ['W2V Propio', 'W2V Pretrained', 'TF-IDF']
colors = ['#2ecc71', '#3498db', '#e74c3c']  # Verde, Azul, Rojo

# Extraer solo scores de test para cada approach
scores_test = [
    [score[1] for score in f1_scores_1],  # Test scores w2v propio
    [score[1] for score in f1_scores_2],  # Test scores w2v pretrained
    [score[1] for score in f1_scores_3]   # Test scores tfidf
]

# Configurar el gráfico
x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

# Crear barras para cada approach
for i, (approach, scores) in enumerate(zip(approaches, scores_test)):
    offset = (i - 1) * width  # Centrar las barras: -width, 0, +width
    bars = ax.bar(x + offset, scores, width, label=approach, color=colors[i], alpha=0.8)

    # Añadir valores en las barras
    for bar, score in zip(bars, scores):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Modelos', fontsize=12)
ax.set_ylabel('F1-Score (Macro)', fontsize=12)
ax.set_title('Comparación de estrategias\nF1-Score en Test', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
ax.set_ylim([0, 1])
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Topic Modeling

In [ ]:
!gdown 1b0cmrHesxIXzZhxAEqRizdL8zOFXyK9V

In [ ]:
import pandas as pd

df = pd.read_csv('topics_unlabeled.csv')
df

In [ ]:
from nltk.corpus import stopwords
import string
from nltk.tokenize import word_tokenize
import re

STOPWORDS = set(stopwords.words('english'))

def clean_text(text: str) -> list[str]:
    """
    Limpia un documento y retorna lista de tokens lista para W2V.
    Pipeline:
      1. Eliminar HTML tags
      2. Eliminar URLs y emails
      3. Lowercase
      4. Eliminar puntuación y caracteres no-ASCII
      5. Normalizar espacios
      6. Tokenizar
      7. Eliminar stopwords y tokens no alfabéticos
    """
    if not isinstance(text, str):
        return []

    # 1. HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 2. URLs y emails
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)

    # 3. Lowercase
    text = text.lower()

    # 4. Puntuación y caracteres no-ASCII
    text = text.encode('ascii', errors='ignore').decode()
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))

    # 5. Normalizar espacios
    text = re.sub(r'\s+', ' ', text).strip()

    # 6. Tokenizar
    tokens = word_tokenize(text)

    # 7. Stopwords y tokens no alfabéticos
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS]

    return tokens

def clean_dataset(df, output_path: str = None, text_col: str = 'text'):
    """
    Aplica clean_text y agrega columna 'tokens' con la lista de tokens limpios.
    """
    print(f"Shape original: {df.shape}")

    # Eliminar nulos y duplicados
    df = df.dropna(subset=[text_col]).drop_duplicates(subset=[text_col]).reset_index(drop=True)
    print(f"Shape tras drop nulos/duplicados: {df.shape}")

    df['tokens'] = df[text_col].apply(clean_text)

    # Filtrar documentos que quedaron vacíos tras la limpieza
    empty_mask = df['tokens'].apply(len) == 0
    print(f"Documentos vacíos tras limpieza: {empty_mask.sum()} → eliminados")
    df = df[~empty_mask].reset_index(drop=True)

    print(f"Shape final: {df.shape}")

    return df

In [ ]:
df = clean_dataset(df)
df

In [ ]:
tokenized_docs = list(df['tokens'].values)

In [ ]:
from gensim.models import KeyedVectors

pretrained_model_path = 'GoogleNews-vectors-negative300.bin.gz'

pt_w2v_model = KeyedVectors.load_word2vec_format(pretrained_model_path, binary=True)

In [ ]:
vector_size = pt_w2v_model.vector_size
doc_vectors = np.zeros((len(tokenized_docs), vector_size))

zero_vector_num = 0

for i, doc in enumerate(tokenized_docs):
    try:
        doc_vectors[i] = pt_w2v_model.get_mean_vector(doc)
    except:
        zero_vector_num += 1

print(f"Vectores nulos en el corpus: {zero_vector_num}")

In [ ]:
from sklearn.cluster import KMeans

num_clusters = 8

kmeans = KMeans(n_clusters=num_clusters, random_state=642)
kmeans.fit(doc_vectors)

In [ ]:
from sklearn.metrics import silhouette_score

labels = kmeans.labels_

silhouette_score(doc_vectors, labels)

In [ ]:
df['cluster'] = labels
df

In [ ]:
docs_per_cluster = {j: [] for j in range(num_clusters)}

for cluster in range(num_clusters):
    cluster_docs = df[df['cluster']==cluster]['tokens'].values
    cluster_docs = [" ".join(doc) for doc in cluster_docs]
    docs_per_cluster[cluster] = cluster_docs
    print(f"Cluster{cluster}: {len(cluster_docs)} documentos")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, num_clusters, figsize=(5*num_clusters, 5))
for cluster, ax in zip(range(num_clusters), axs.flatten()):
    text = " ".join(docs_per_cluster[cluster])

    # Crear la nube de palabras
    wc = WordCloud(
        background_color='white',
        width=800,
        height=400,
        max_words=100,
        colormap='viridis',
        random_state=42
    ).generate(text)

    # Mostrar la nube de palabras en el subplot
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Cluster {cluster}', fontsize=16, fontweight='bold')
    ax.axis('off')

# Ajustar el espaciado entre subplots
plt.tight_layout()
plt.show()

In [ ]:
nums = list(range(2,10))
siluetas = []

for num in nums:
    kmeans = KMeans(n_clusters=num, random_state=642)
    kmeans.fit(doc_vectors)
    silueta = silhouette_score(doc_vectors, kmeans.labels_)
    siluetas.append(silueta)

plt.figure()
plt.plot(nums, siluetas)
plt.xlabel('Número de clusters')
plt.ylabel('Silueta')
plt.title("Silueta vs Número de clusters")
plt.show()